# Task 5: UDPipe (SynTagRus и Taiga)

Синтаксический разбор предложения:

> По сообщению полиции, она убила Андре Прайса III, прижав его лицо к надувному матрасу в гостиной перед попыткой сделать то же самое с ее дочерью Энджел.

Ниже:
- загружается эталонный `ru_pud-ud-test.conllu`;
- выполняется парсинг через UDPipe API моделями SynTagRus и Taiga;
- считается UAS/LAS относительно эталона;
- вычисляется глубина дерева.

In [1]:
%pip -q install conllu requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from collections import deque

import requests
from conllu import parse

DATASET_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Text_Processing/Task_5/ru_pud-ud-test.conllu"
UDPIPE_API = "https://lindat.mff.cuni.cz/services/udpipe/api"
TARGET_TEXT = "По сообщению полиции, она убила Андре Прайса III, прижав его лицо к надувному матрасу в гостиной перед попыткой сделать то же самое с ее дочерью Энджел."

In [3]:
def download_gold_sentences(url: str):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return parse(response.text)


def keep_word_tokens(sentence):
    # В CoNLL-U есть служебные строки и составные токены, исключаем их.
    return [tok for tok in sentence if isinstance(tok["id"], int)]


def find_gold_sentence(sentences, text: str):
    for sent in sentences:
        if sent.metadata.get("text") == text:
            return sent
    raise ValueError("Предложение не найдено в эталонном файле")

In [4]:
def select_latest_model(prefix: str) -> str:
    response = requests.get(f"{UDPIPE_API}/models", timeout=60)
    response.raise_for_status()
    payload = response.json()

    candidates = [name for name in payload["models"] if name.startswith(prefix)]
    if not candidates:
        raise ValueError(f"Модель с префиксом {prefix!r} не найдена")

    # Последние версии имеют больший номер UD и более поздний суффикс даты.
    return sorted(candidates, reverse=True)[0]


def parse_with_udpipe(text: str, model_name: str):
    response = requests.post(
        f"{UDPIPE_API}/process",
        data={
            "model": model_name,
            "data": text,
            "tokenizer": "1",
            "tagger": "1",
            "parser": "1",
        },
        timeout=60,
    )
    response.raise_for_status()

    try:
        payload = response.json()
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"UDPipe вернул не-JSON ответ: {response.text[:200]}") from exc

    if "result" not in payload:
        raise RuntimeError(f"UDPipe не вернул поле 'result': {payload}")

    parsed = parse(payload["result"])
    if not parsed:
        raise RuntimeError("UDPipe вернул пустой результат")
    return parsed[0]

In [5]:
def edge_sets(sentence):
    tokens = keep_word_tokens(sentence)
    unlabeled = {(tok["head"], tok["id"]) for tok in tokens}
    labeled = {(tok["head"], tok["id"], tok["deprel"]) for tok in tokens}
    return labeled, unlabeled


def uas_las(gold_sentence, pred_sentence):
    gold_labeled, gold_unlabeled = edge_sets(gold_sentence)
    pred_labeled, pred_unlabeled = edge_sets(pred_sentence)

    uas = len(gold_unlabeled & pred_unlabeled) / len(gold_unlabeled)
    las = len(gold_labeled & pred_labeled) / len(gold_labeled)
    return uas, las


def tree_depth(sentence):
    tokens = keep_word_tokens(sentence)
    children = {tok["id"]: [] for tok in tokens}
    roots = []

    for tok in tokens:
        if tok["head"] == 0:
            roots.append(tok["id"])
        else:
            children.setdefault(tok["head"], []).append(tok["id"])

    max_depth = 0
    queue = deque((root, 0) for root in roots)

    while queue:
        node, depth = queue.popleft()
        max_depth = max(max_depth, depth)
        for child in children.get(node, []):
            queue.append((child, depth + 1))

    return max_depth


def print_dependency_tree(sentence):
    tokens = keep_word_tokens(sentence)
    by_id = {tok["id"]: tok for tok in tokens}
    children = {tok["id"]: [] for tok in tokens}
    roots = []

    for tok in tokens:
        if tok["head"] == 0:
            roots.append(tok["id"])
        else:
            children.setdefault(tok["head"], []).append(tok["id"])

    for key in children:
        children[key].sort()

    def walk(node_id, level=0):
        node = by_id[node_id]
        print("  " * level + f"{node['form']} [{node['deprel']}]")
        for child_id in children.get(node_id, []):
            walk(child_id, level + 1)

    for root in roots:
        walk(root)

In [6]:
gold_sentences = download_gold_sentences(DATASET_URL)
gold_sentence = find_gold_sentence(gold_sentences, TARGET_TEXT)

gold_sentence.metadata

{'sent_id': 'n01011011',
 'parallel_id': 'pud/n01011011',
 'text': 'По сообщению полиции, она убила Андре Прайса III, прижав его лицо к надувному матрасу в гостиной перед попыткой сделать то же самое с ее дочерью Энджел.',
 'english_text': 'She killed Andre Price III by pressing his face into an air mattress in her sitting room before trying to do the same to her daughter, Angel, police said.'}

In [7]:
syn_model = select_latest_model("russian-syntagrus")
syn_sentence = parse_with_udpipe(TARGET_TEXT, syn_model)

syn_uas, syn_las = uas_las(gold_sentence, syn_sentence)
syn_depth = tree_depth(syn_sentence)

print("SynTagRus model:", syn_model)
print("UAS:", round(syn_uas, 3))
print("LAS:", round(syn_las, 3))
print("Depth:", syn_depth)
print("\nTree:")
print_dependency_tree(syn_sentence)

SynTagRus model: russian-syntagrus-ud-2.6-200830
UAS: 0.931
LAS: 0.828
Depth: 6

Tree:
убила [root]
  сообщению [parataxis]
    По [case]
    полиции [nmod]
    , [punct]
  она [nsubj]
  Андре [obj]
    Прайса [flat:name]
    III [nummod]
  прижав [advcl]
    , [punct]
    лицо [obj]
      его [det]
    матрасу [obl]
      к [case]
      надувному [amod]
      гостиной [nmod]
        в [case]
    попыткой [obl]
      перед [case]
      сделать [obl]
        то [obj]
          же [advmod]
          самое [amod]
          дочерью [nmod]
            с [case]
            ее [nmod]
            Энджел [appos]
  . [punct]


In [8]:
taiga_model = select_latest_model("russian-taiga")
taiga_sentence = parse_with_udpipe(TARGET_TEXT, taiga_model)

taiga_uas, taiga_las = uas_las(gold_sentence, taiga_sentence)
taiga_depth = tree_depth(taiga_sentence)

print("Taiga model:", taiga_model)
print("UAS:", round(taiga_uas, 3))
print("LAS:", round(taiga_las, 3))
print("Depth:", taiga_depth)
print("\nTree:")
print_dependency_tree(taiga_sentence)

Taiga model: russian-taiga-ud-2.6-200830
UAS: 0.966
LAS: 0.897
Depth: 5

Tree:
убила [root]
  сообщению [parataxis]
    По [case]
    полиции [nmod]
    , [punct]
  она [nsubj]
  Андре [obj]
    Прайса [flat:name]
    III [amod]
  прижав [advcl]
    , [punct]
    лицо [obj]
      его [det]
    матрасу [obl]
      к [case]
      надувному [amod]
      гостиной [nmod]
        в [case]
    попыткой [obl]
      перед [case]
      сделать [obl]
        то [obj]
          же [advmod]
          самое [amod]
        дочерью [obl]
          с [case]
          ее [det]
          Энджел [appos]
  . [punct]


In [9]:
print("Ответы для формы:")
print("SynTagRus UAS:", round(syn_uas, 3))
print("SynTagRus LAS:", round(syn_las, 3))
print("SynTagRus depth:", syn_depth)
print("Taiga UAS:", round(taiga_uas, 3))
print("Taiga LAS:", round(taiga_las, 3))
print("Taiga depth:", taiga_depth)

Ответы для формы:
SynTagRus UAS: 0.931
SynTagRus LAS: 0.828
SynTagRus depth: 6
Taiga UAS: 0.966
Taiga LAS: 0.897
Taiga depth: 5
